# SHANKS — duas GPUs pelo Jupyter, Spark e YARN

Este notebook solicita dois executores YARN, cada um com uma GPU, valida o ambiente remoto e usa `TorchDistributor` para executar um treino PyTorch DDP com NCCL. Execute as células em ordem.

O treino usa dados sintéticos e mede computação e comunicação. Ele não representa a qualidade de um modelo de produção.


In [ ]:
import importlib
import json
import pyspark
from pathlib import Path
from pyspark.ml.torch.distributor import TorchDistributor

for package in ("torch", "numpy"):
    importlib.import_module(package)
print("PySpark:", pyspark.__version__)
print("TorchDistributor disponível.")


In [ ]:
import os
from pyspark.sql import SparkSession

EXECUTOR_PYTHON = "/opt/shanks-venv/bin/python"
discovery = Path("shanks_discover_gpu.sh").resolve()
discovery.write_text("""#!/usr/bin/env python3
import json
import subprocess
result = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"], text=True)
addresses = [line.strip() for line in result.splitlines() if line.strip()]
if not addresses:
    raise RuntimeError("Nenhuma GPU detectada pelo nvidia-smi")
print(json.dumps({"name": "gpu", "addresses": addresses}))
""")
discovery.chmod(0o755)

active = SparkSession.getActiveSession()
if active is not None:
    active.stop()
elif "spark" in globals():
    spark.stop()

os.environ["PYSPARK_PYTHON"] = EXECUTOR_PYTHON
spark = (SparkSession.builder.master("yarn")
    .appName("SHANKS - PyTorch DDP - 2 GPUs")
    .config("spark.submit.deployMode", "client")
    .config("spark.executor.instances", "2")
    .config("spark.executor.cores", "2")
    .config("spark.task.cpus", "2")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.memoryOverhead", "1g")
    .config("spark.yarn.am.memory", "512m")
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.speculation", "false")
    .config("spark.pyspark.python", EXECUTOR_PYTHON)
    .config("spark.executor.resource.gpu.amount", "1")
    .config("spark.task.resource.gpu.amount", "1")
    .config("spark.executor.resource.gpu.discoveryScript", "./shanks_discover_gpu.sh")
    .config("spark.files", str(discovery))
    .config("spark.executorEnv.NCCL_SOCKET_IFNAME", "ztu")
    .config("spark.executorEnv.NCCL_IB_DISABLE", "1")
    .config("spark.executorEnv.OMP_NUM_THREADS", "2")
    .getOrCreate())
print("Aplicação YARN:", spark.sparkContext.applicationId)
print("Spark:", spark.version)


## Verificar os dois executores

Esta etapa confirma que duas máquinas diferentes possuem PyTorch CUDA/NCCL funcional. Se ficar aguardando recursos, confira se o YARN anuncia duas GPUs livres.


In [ ]:
def verificar_particao(iterator):
    import json, os, socket, subprocess, sys
    from pyspark import BarrierTaskContext
    ctx = BarrierTaskContext.get()
    try:
        addresses = ctx.resources()["gpu"].addresses
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = ",".join(addresses)
        probe = """import json, torch
assert torch.cuda.is_available(), 'CUDA indisponível neste executor'
assert torch.distributed.is_nccl_available(), 'NCCL indisponível'
x = torch.randn(512, 512, device='cuda'); y = x @ x
torch.cuda.synchronize()
print(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda,
 'gpu': torch.cuda.get_device_name(0),
 'vram_gib': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)}))
"""
        run = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True, timeout=120)
        if run.returncode:
            raise RuntimeError(run.stderr[-4000:])
        info = json.loads(run.stdout.strip().splitlines()[-1])
        info.update(host=socket.gethostname(), gpu_addresses=addresses, python=sys.executable, ok=True)
    except Exception as error:
        info = {"host": socket.gethostname(), "ok": False, "error": str(error)}
    ctx.allGather(json.dumps(info))
    yield info

workers = (spark.sparkContext.parallelize([0, 1], 2).barrier()
           .mapPartitions(verificar_particao).collect())
print(json.dumps(workers, indent=2, ensure_ascii=False))
assert all(w["ok"] for w in workers)
assert len({w["host"] for w in workers}) == 2
assert len({w["torch"] for w in workers}) == 1


In [ ]:
def treinar_gpu(steps=200, global_batch=1024, width=512, depth=4, warmup=10, seed=42):
    import json, os, platform, socket, time
    from datetime import datetime, timezone, timedelta
    from types import SimpleNamespace
    import torch
    import torch.distributed as dist
    from torch import nn
    from torch.nn.parallel import DistributedDataParallel as DDP
    args = SimpleNamespace(steps=steps, global_batch=global_batch, width=width, depth=depth, warmup=warmup, seed=seed)
    if min(steps, global_batch, width, depth) < 1 or warmup < 0:
        raise ValueError("Parâmetros inválidos")
    if not torch.cuda.is_available() or not dist.is_nccl_available():
        raise RuntimeError("CUDA ou NCCL indisponível")
    rank = int(os.environ.get("RANK", "0")); world = int(os.environ.get("WORLD_SIZE", "1")); local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    if args.global_batch % world:
        raise ValueError("global_batch deve ser divisível pela quantidade de GPUs")
    torch.cuda.set_device(local_rank); device = torch.device("cuda", local_rank)
    dist.init_process_group("nccl", timeout=timedelta(minutes=10))
    try:
        torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)
        torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
        layers = [nn.Linear(128, args.width), nn.ReLU()]
        for _ in range(args.depth - 1): layers.extend([nn.Linear(args.width, args.width), nn.ReLU()])
        layers.append(nn.Linear(args.width, 2))
        raw_model = nn.Sequential(*layers).to(device); parameters = sum(p.numel() for p in raw_model.parameters())
        model = DDP(raw_model, device_ids=[local_rank]) if world > 1 else raw_model
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001); criterion = nn.CrossEntropyLoss()
        generator = torch.Generator(device="cpu").manual_seed(args.seed + 1); size = max(16384, args.global_batch * 4)
        data = torch.randn(size, 128, generator=generator).to(device); labels = (data[:, :8].sum(dim=1) > 0).long()
        local_batch = args.global_batch // world; offsets = torch.arange(local_batch, device=device) + rank * local_batch
        def batch(step):
            indexes = (offsets + step * args.global_batch) % size
            return data[indexes], labels[indexes]
        def train_step(step):
            x, y = batch(step); optimizer.zero_grad(set_to_none=True); loss = criterion(model(x), y); loss.backward(); optimizer.step(); return loss.detach()
        @torch.no_grad()
        def evaluate():
            x, y = batch(0); logits = model(x); values = torch.stack([criterion(logits, y), (logits.argmax(1) == y).float().mean()]); dist.all_reduce(values); return (values / world).cpu().tolist()
        metadata = {"rank": rank, "host": socket.gethostname(), "gpu": torch.cuda.get_device_name(device), "torch": torch.__version__, "cuda": torch.version.cuda, "python": platform.python_version(), "local_batch": local_batch}
        initial_loss, initial_acc = evaluate()
        for step in range(args.warmup): train_step(step)
        torch.cuda.synchronize(); dist.barrier(); torch.cuda.reset_peak_memory_stats(); start = time.perf_counter()
        for step in range(args.steps): train_step(args.warmup + step)
        torch.cuda.synchronize(); elapsed_tensor = torch.tensor(time.perf_counter() - start, dtype=torch.float64, device=device); dist.all_reduce(elapsed_tensor, op=dist.ReduceOp.MAX); seconds = elapsed_tensor.item()
        final_loss, final_acc = evaluate(); max_difference = torch.zeros((), device=device)
        if world > 1:
            for parameter in raw_model.parameters():
                reference = parameter.detach().clone(); dist.broadcast(reference, src=0); max_difference = torch.maximum(max_difference, (parameter - reference).abs().max())
            dist.all_reduce(max_difference, op=dist.ReduceOp.MAX)
        workers = [None] * world; dist.all_gather_object(workers, metadata)
        if rank == 0:
            return {"experiment": "SHANKS PyTorch DDP", "timestamp_utc": datetime.now(timezone.utc).isoformat(), "gpus": world, "parameters": parameters, "config": vars(args), "seconds": round(seconds, 4), "samples_per_second": round(args.steps * args.global_batch / seconds, 2), "initial_probe_loss": initial_loss, "final_probe_loss": final_loss, "initial_probe_accuracy": initial_acc, "final_probe_accuracy": final_acc, "max_parameter_difference_between_ranks": max_difference.item(), "workers": workers}
        return None
    finally:
        dist.destroy_process_group()


In [ ]:
config_treino = dict(steps=200, global_batch=1024, width=512, depth=4, warmup=10, seed=42)
resultado_duas = TorchDistributor(num_processes=2, local_mode=False, use_gpu=True).run(treinar_gpu, **config_treino)
print(json.dumps(resultado_duas, indent=2, ensure_ascii=False))
assert resultado_duas["gpus"] == 2
assert len({w["host"] for w in resultado_duas["workers"]}) == 2
assert resultado_duas["max_parameter_difference_between_ranks"] < 1e-6
Path("shanks_duas_gpus.json").write_text(json.dumps(resultado_duas, indent=2, ensure_ascii=False) + "\n")


## Diagnóstico

Se os executores não iniciarem, confira GPUs livres no YARN, memória, vcores e os logs da aplicação. Se o NCCL travar, confira a interface ZeroTier configurada em `NCCL_SOCKET_IFNAME` e a conectividade entre os nós. Ao terminar, execute `spark.stop()`.
